# 👤 Módulo 2 — Ingreso de Personal
**Sistema de Gestión de Recursos Humanos**

Este notebook permite registrar un nuevo colaborador en dos pasos:
1. **Datos personales** → tabla `empleados`
2. **Datos laborales** → tabla `personal`


In [ ]:
# ── 1. Librerías ──────────────────────────────────────────────────────────────
import mysql.connector
import random
import string
from datetime import date, datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


In [ ]:
# Conexion a la base de datos
import os
import mysql.connector

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': int(os.getenv('DB_PORT', '1989')),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'rrhh_sistema'),
    'charset': 'utf8mb4',
}

def get_conn():
    return mysql.connector.connect(**DB_CONFIG)

try:
    conn = get_conn()
    conn.close()
    print(f'Conexion exitosa -> {DB_CONFIG["database"]} (Puerto {DB_CONFIG["port"]})')
except mysql.connector.Error as e:
    print(f'Error de conexion: {e}')



In [ ]:
# ── 3. Funciones auxiliares ───────────────────────────────────────────────────

def generar_codigo() -> str:
    """Genera un código de empresa de 5 caracteres alfanuméricos único."""
    conn = get_conn()
    cur  = conn.cursor()
    while True:
        code = ''.join(random.choices(string.ascii_uppercase + string.digits, k=5))
        cur.execute("SELECT 1 FROM empleados WHERE codigo_empresa = %s", (code,))
        if not cur.fetchone():
            break
    cur.close()
    conn.close()
    return code

def cedula_existe(cedula: str) -> bool:
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute("SELECT 1 FROM empleados WHERE numero_cedula = %s", (cedula,))
    existe = cur.fetchone() is not None
    cur.close(); conn.close()
    return existe

def get_departamentos() -> list[tuple]:
    """Devuelve [(id, nombre), ...] de la tabla departamentos."""
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute("SELECT id_departamento, nombre_departamento FROM departamentos ORDER BY nombre_departamento")
    rows = cur.fetchall()
    cur.close(); conn.close()
    return rows

def insertar_empleado(data: dict) -> str:
    """Inserta en empleados y devuelve el codigo_empresa generado."""
    codigo = generar_codigo()
    sql = """
        INSERT INTO empleados
            (codigo_empresa, numero_cedula, nombre, apellido,
             fecha_nacimiento, nacionalidad, direccion,
             telefono_principal, telefono_secundario, estado)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,'activo')
    """
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute(sql, (
        codigo,
        data["cedula"],
        data["nombre"],
        data["apellido"],
        data["fecha_nac"],
        data["nacionalidad"],
        data["direccion"],
        data["tel_principal"],
        data["tel_secundario"] or None,
    ))
    conn.commit()
    cur.close(); conn.close()
    return codigo

def insertar_personal(codigo: str, data: dict):
    """Inserta en personal los datos laborales del empleado."""
    sql = """
        INSERT INTO personal
            (codigo_empresa, fecha_ingreso, puesto, id_departamento, observaciones)
        VALUES (%s, %s, %s, %s, %s)
    """
    conn = get_conn()
    cur  = conn.cursor()
    cur.execute(sql, (
        codigo,
        data["fecha_ingreso"],
        data["puesto"],
        data["id_departamento"],
        data["observaciones"] or None,
    ))
    conn.commit()
    cur.close(); conn.close()

print("✅  Funciones auxiliares cargadas.")


In [ ]:
# ── 4+5. FORMULARIO COMPLETO — Paso 1 (empleados) + Paso 2 (personal) ──────────

out1 = widgets.Output()
out2 = widgets.Output()

# ── PASO 1: Datos Personales ─────────────────────────────────────────────────
lbl_s1 = widgets.HTML("<h3 style='color:#1a73e8'>📋 Paso 1 de 2 · Datos Personales</h3>")

w_nombre      = widgets.Text(description="Nombre *",        placeholder="Ej. María",          layout=widgets.Layout(width="45%"))
w_apellido    = widgets.Text(description="Apellido *",       placeholder="Ej. González",       layout=widgets.Layout(width="45%"))
w_cedula      = widgets.Text(description="Cédula *",         placeholder="Ej. 8-123-4567",     layout=widgets.Layout(width="45%"))
w_nacimiento  = widgets.DatePicker(description="Nacimiento *",                                  layout=widgets.Layout(width="45%"))
w_nacional    = widgets.Text(description="Nacionalidad *",   placeholder="Ej. Panameña",       layout=widgets.Layout(width="45%"))
w_direccion   = widgets.Textarea(description="Dirección *",  placeholder="Calle, sector...",   layout=widgets.Layout(width="93%"), rows=2)
w_tel1        = widgets.Text(description="Tel. principal *", placeholder="Ej. 6000-0000",      layout=widgets.Layout(width="45%"))
w_tel2        = widgets.Text(description="Tel. secundario",  placeholder="(opcional)",         layout=widgets.Layout(width="45%"))
btn_paso1     = widgets.Button(description="Continuar →", button_style="primary",
                               layout=widgets.Layout(width="180px", height="36px"))

form1 = widgets.VBox([
    lbl_s1,
    widgets.HBox([w_nombre, w_apellido]),
    widgets.HBox([w_cedula, w_nacimiento]),
    widgets.HBox([w_nacional]),
    w_direccion,
    widgets.HBox([w_tel1, w_tel2]),
    btn_paso1, out1
], layout=widgets.Layout(padding="16px", border="1px solid #ddd",
                          border_radius="8px", width="700px"))

# ── PASO 2: Datos Laborales ───────────────────────────────────────────────────
lbl_s2 = widgets.HTML("<h3 style='color:#1a73e8'>🏢 Paso 2 de 2 · Datos Laborales</h3>")

deptos     = get_departamentos()
opts_deptos = {nombre: id_ for id_, nombre in deptos} if deptos else {}

if deptos:
    w_depto = widgets.Dropdown(description="Departamento *",
                               options=list(opts_deptos.keys()),
                               layout=widgets.Layout(width="45%"))
else:
    w_depto = widgets.Dropdown(description="Departamento *",
                               options=["(sin departamentos — agrégalos primero)"],
                               layout=widgets.Layout(width="45%"))

w_puesto        = widgets.Text(description="Puesto *", placeholder="Ej. Analista de RRHH",
                                layout=widgets.Layout(width="45%"))
w_fecha_ingreso = widgets.DatePicker(description="Fecha ingreso *",
                                      layout=widgets.Layout(width="45%"))
w_observaciones = widgets.Textarea(description="Observaciones", placeholder="(opcional)",
                                    layout=widgets.Layout(width="93%"), rows=3)
btn_guardar     = widgets.Button(description="💾 Guardar Empleado", button_style="success",
                                  layout=widgets.Layout(width="200px", height="36px"))

# form2 definido ANTES de usarlo en on_paso1
form2 = widgets.VBox([
    lbl_s2,
    widgets.HBox([w_depto, w_puesto]),
    widgets.HBox([w_fecha_ingreso]),
    w_observaciones,
    btn_guardar, out2
], layout=widgets.Layout(padding="16px", border="1px solid #ddd",
                          border_radius="8px", width="700px",
                          margin_top="12px", display="none"))

# ── Estado compartido ─────────────────────────────────────────────────────────
_state = {}

def on_paso1(b):
    with out1:
        clear_output()
        errores = []
        if not w_nombre.value.strip():     errores.append("⚠ Nombre requerido.")
        if not w_apellido.value.strip():   errores.append("⚠ Apellido requerido.")
        if not w_cedula.value.strip():     errores.append("⚠ Cédula requerida.")
        if not w_nacimiento.value:         errores.append("⚠ Fecha de nacimiento requerida.")
        if not w_nacional.value.strip():   errores.append("⚠ Nacionalidad requerida.")
        if not w_direccion.value.strip():  errores.append("⚠ Dirección requerida.")
        if not w_tel1.value.strip():       errores.append("⚠ Teléfono principal requerido.")

        if w_cedula.value.strip() and cedula_existe(w_cedula.value.strip()):
            errores.append("❌ Ya existe un empleado con esa cédula.")

        if errores:
            for e in errores: print(e)
            return

        _state["paso1"] = {
            "nombre":         w_nombre.value.strip(),
            "apellido":       w_apellido.value.strip(),
            "cedula":         w_cedula.value.strip(),
            "fecha_nac":      w_nacimiento.value,
            "nacionalidad":   w_nacional.value.strip(),
            "direccion":      w_direccion.value.strip(),
            "tel_principal":  w_tel1.value.strip(),
            "tel_secundario": w_tel2.value.strip(),
        }
        print("✅  Datos personales validados. Completa el Paso 2 ↓")
        form2.layout.display = ""   # ahora form2 ya existe → sin error

def on_guardar(b):
    with out2:
        clear_output()
        if "paso1" not in _state:
            print("❌ Primero completa el Paso 1."); return
        if not opts_deptos:
            print("❌ No hay departamentos en la BD."); return

        errores = []
        if not w_puesto.value.strip():  errores.append("⚠ Puesto requerido.")
        if not w_fecha_ingreso.value:   errores.append("⚠ Fecha de ingreso requerida.")
        if errores:
            for e in errores: print(e)
            return

        try:
            codigo = insertar_empleado(_state["paso1"])
            insertar_personal(codigo, {
                "fecha_ingreso":   w_fecha_ingreso.value,
                "puesto":          w_puesto.value.strip(),
                "id_departamento": opts_deptos[w_depto.value],
                "observaciones":   w_observaciones.value.strip(),
            })
            nombre_completo = f"{_state['paso1']['nombre']} {_state['paso1']['apellido']}"
            display(HTML(f"""
            <div style='background:#e8f5e9;border:1px solid #66bb6a;border-radius:8px;padding:16px'>
                <h4 style='color:#2e7d32;margin:0'>✅ Empleado registrado exitosamente</h4>
                <table style='margin-top:10px;border-collapse:collapse'>
                    <tr><td style='padding:4px 12px 4px 0'><b>Nombre:</b></td><td>{nombre_completo}</td></tr>
                    <tr><td><b>Código empresa:</b></td><td><code>{codigo}</code></td></tr>
                    <tr><td><b>Cédula:</b></td><td>{_state['paso1']['cedula']}</td></tr>
                    <tr><td><b>Puesto:</b></td><td>{w_puesto.value.strip()}</td></tr>
                    <tr><td><b>Departamento:</b></td><td>{w_depto.value}</td></tr>
                    <tr><td><b>Fecha de ingreso:</b></td><td>{w_fecha_ingreso.value}</td></tr>
                </table>
            </div>
            """))
            _state.clear()
        except Exception as ex:
            print(f"❌ Error al guardar: {ex}")

btn_paso1.on_click(on_paso1)
btn_guardar.on_click(on_guardar)

display(form1, form2)


In [ ]:
# ── 6. Consulta rápida — últimos empleados ingresados ────────────────────────
import pandas as pd

try:
    conn = get_conn()
    df = pd.read_sql("""
        SELECT e.codigo_empresa, e.nombre, e.apellido, e.numero_cedula,
               p.puesto, d.nombre_departamento, p.fecha_ingreso, e.estado
        FROM   empleados e
        JOIN   personal  p ON p.codigo_empresa = e.codigo_empresa
        JOIN   departamentos d ON d.id_departamento = p.id_departamento
        ORDER  BY p.fecha_ingreso DESC
        LIMIT  20
    """, conn)
    conn.close()
    display(HTML("<h3 style='color:#1a73e8'>📊 Últimos empleados registrados</h3>"))
    display(df.style.set_table_styles([
        {'selector':'th','props':[('background','#1a73e8'),('color','white'),('padding','6px 12px')]},
        {'selector':'td','props':[('padding','5px 12px')]}
    ]).hide(axis='index'))
except Exception as ex:
    print(f"Error al consultar: {ex}")
